# W02 — ML Task Framing

**Lane:** Titanic Survival Prediction

**Approach:** Core first, AI second — the framing below is reasoned out by hand, then made real in code with a simple, interpretable baseline model (Logistic Regression).

## 1) My lane as an ML task (type)

**Task type: Classification (binary).**

My lane is Titanic passenger data. The question I want to answer for each passenger is *"did this person survive or not?"* — a yes/no outcome. That makes this a **binary classification** problem, not clustering (I'm not looking for unlabeled groupings), not ranking (I don't need a relative order between passengers), and not scoring/regression (the target isn't a continuous number).

## 2) Target or proxy

**Target: `survived`** — a binary column already present in the dataset (0 = did not survive, 1 = survived).

This is a *true label*, not a proxy: Titanic's outcome is historical fact, recorded directly, so I don't need to approximate it with a stand-in signal (the way, say, "clicked" might proxy for "interested"). Each row already carries the ground truth I'm trying to predict.

## 3) Success metric

**Primary metric: Accuracy**, supported by a look at the **confusion matrix** (precision/recall) since the classes aren't perfectly balanced (more people died than survived).

- Accuracy answers the plain question: *of all passengers, how many did I classify correctly?* — a reasonable headline number for a first pass.
- Because a model could get decent accuracy just by always predicting "did not survive," I also check recall on the survived class, so I know I'm not fooling myself with a lazy majority-class model.

## 4) The unit of analysis, as a real dataframe

**One row = one passenger** aboard the Titanic. Each row holds that passenger's ticket class, sex, age, fare, family aboard, port of embarkation, and whether they survived. Below I load the data and show it as an actual dataframe.

In [1]:
import seaborn as sns
import pandas as pd

# Load the Titanic dataset that ships with seaborn
titanic = sns.load_dataset('titanic')

print(f"Shape: {titanic.shape[0]} rows (passengers) x {titanic.shape[1]} columns (features)")
titanic.head()

Shape: 891 rows (passengers) x 15 columns (features)


,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [2]:
# Quick look at the unit of analysis: one row = one passenger
titanic[['pclass', 'sex', 'age', 'fare', 'sibsp', 'parch', 'embarked', 'survived']].sample(5, random_state=42)

,pclass,sex,age,fare,sibsp,parch,embarked,survived
709,3,male,NaN,15.2458,1,1,C,1
439,2,male,31.0,10.5000,0,0,S,0
840,3,male,20.0,7.9250,0,0,S,0
720,2,female,6.0,33.0000,0,1,S,1
39,3,female,14.0,11.2417,1,0,C,1


## 5) Why ML beats a fixed rule here

I could try a fixed rule like *"predict survived if sex == female"* — and it would actually do okay, because sex is a strong single signal on the Titanic (women and children first). But survival really depended on the **combination** of several factors at once: class (proxy for deck location and access to lifeboats), age, fare (also a class proxy), and family size (traveling alone vs. with kin). A fixed rule can only ever encode one or two of these thresholds by hand, and it can't learn the right *weights and interactions* between them from data. A model like logistic regression can combine all these variables, learn how much each one matters, and adjust its boundary as the mix of factors changes per passenger — which is exactly the kind of pattern a fixed if/else rule can't capture well.

In [3]:
# Confirm sex alone (a plausible "fixed rule") isn't the whole story
titanic.groupby('sex')['survived'].mean()

sex
female    0.742038
male      0.188908
Name: survived, dtype: float64

A rule of *"predict survived if female"* gets close, but it ignores class, age, and fare differences within each sex group — that's the gap ML can close.

## 6) Self-check

- **Task type named?** ✅ Binary classification.
- **Target/proxy named?** ✅ `survived`, a true historical label (not a proxy).
- **Success metric named?** ✅ Accuracy, checked against the confusion matrix/recall.
- **Unit of analysis shown as a real dataframe?** ✅ One row = one passenger, shown above.
- **Explained in my own words why this is ML and not just a rule?** ✅ Multiple interacting factors, not one clean threshold.
- **Tied to a real action?** ✅ Below, I train a simple baseline model and see how it actually performs, to check the framing holds up.

## Making it real: a simple baseline model (Logistic Regression)

This isn't the deliverable's main point (that's the framing above) — it's a quick, honest sanity check that the framing is buildable.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# A small, simple feature set — enough to test the framing, not a full feature-engineering pass
features = ['pclass', 'sex', 'age', 'fare', 'sibsp', 'parch', 'embarked']
target = 'survived'

df = titanic[features + [target]].copy()
X = df[features]
y = df[target]

numeric_features = ['age', 'fare', 'sibsp', 'parch']
categorical_features = ['pclass', 'sex', 'embarked']

preprocess = ColumnTransformer([
    ('num', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), numeric_features),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]), categorical_features)
])

model = Pipeline([
    ('preprocess', preprocess),
    ('clf', LogisticRegression(max_iter=1000))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print("\nConfusion matrix (rows=actual, cols=predicted):")
print(confusion_matrix(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=['did not survive', 'survived']))

Accuracy: 0.804

Confusion matrix (rows=actual, cols=predicted):
[[98 12]
 [23 46]]

Classification report:
                 precision    recall  f1-score   support

did not survive       0.81      0.89      0.85       110
       survived       0.79      0.67      0.72        69

       accuracy                           0.80       179
      macro avg       0.80      0.78      0.79       179
   weighted avg       0.80      0.80      0.80       179



### Takeaway

The logistic regression baseline beats a naive "always predict did not survive" rule and also does better than a single "sex == female" rule, because it's combining class, fare, age, and family size along with sex — confirming the framing from section 5: survival depended on multiple interacting factors, which is exactly where ML earns its keep over a fixed rule.